# Introduction
Now that particles have deposited energy in our detector, we move on to "detector simulation". 

For LArTPCs, this means we will:
1. Create some number of ionization electrons along an energy deposition segment
2. Drift those toward our readout anode
3. Simulate the readout of the ionization as they pass by / deposit onto our wires
    - Convolve field and electronics response with the signal (ionization distribution)
    - Add noise
    - Digitize

In DUNE and other LArTPC experiments, we do this (at least parts 2 and 3) using WireCell([1](https://github.com/wirecell/wire-cell-toolkit), [2](https://lar.bnl.gov/wire-cell/)).

Note: this part of the tutorial will be rather simple and mostly an explanation. Let's look at the configuration in `detsim.jsonnet`

## Configuration explanation

### Ionization
After the generation and transport steps, we calculate the number of negative ions produced as a results of the energy deposition

<pre>
    // --- Convert the edep observables' segments to a WCT TrackSegmentSet ---
    segments_to_tracksegmentset: {
      cpp: 'wire_cell_phlex_arrow_edep_segments',
      input_layer: 'event',
    },
    // --- Sample point depos from the segments -----------------------------
    // Ideal lines carry n_electrons directly, so use quanta mode (no recombination).
    sampler: {
      cpp: 'wcph_tracksegmentset_to_deposet',
      inputs: [{ creator: 'segments_to_tracksegmentset', layer: 'event', suffix: 'tracksegmentset' }],
      outputs: [{ suffix: 'deposet' }],
      wct_config: 'tracksegmentset-sampler.jsonnet',
      wct_plugins: ['WireCellGen', 'WireCellPgraph'],
      wct_tla: { ionization: 'recombination' },
    },

</pre>

#### WireCell configuration
WireCell is a big package that itself operates using Data Flow Programming (similar to Phlex). It has its own configuration, as seen in the `wct_config` parameter in the configuration block above.

#### Recombination
Note: part of the configuration references "recombination". This is the process by which ionized electrons reattach to the positive ions within the argon. 

#### Drift and Readout simulation
We then reference steps from WireCell to do the drift and readout simulation portions of the detector simulation
<pre>
    drift: {
      cpp: 'wcph_deposet_filter',
      inputs: [{ creator: 'sampler', layer: 'event', suffix: 'deposet' }],
      outputs: [{ suffix: 'deposet' }],
      wct_config: 'dune/wct/job/drift.jsonnet',
      wct_plugins: ['WireCellGen', 'WireCellAux', 'WireCellPgraph'],
      wct_tla: { detector: 'pdhd', anode_index: '0', input: 'deposet' },
    },
    sim: {
      cpp: 'wcph_deposet_to_frame',
      inputs: [{ creator: 'drift', layer: 'event', suffix: 'deposet' }],
      outputs: [{ suffix: 'frame' }],
      wct_config: 'wc-cfg/sim.jsonnet',
      wct_plugins: ['WireCellGen', 'WireCellSigProc', 'WireCellAux', 'WireCellPgraph'],
      wct_tla: { detector: 'pdhd', anode_index: '0', input: 'drifted' },
    },
</pre>

## Running detsim
Now let's run it!

In [ ]:
# Point to some necessary phlex plugins
import os
os.environ["PHLEX_PLUGIN_PATH"] = f'../build-phlex-examples/:/cvmfs/phaindia.example.org/env/phaindia/view/lib64:{os.environ["PHLEX_PLUGIN_PATH"]}'
! PYTHONPATH=${PWD}/../generate/:${PYTHONPATH}; phlex -c detsim.jsonnet --stage detsim

## Looking at output

In [ ]:
import h5py as h5
f = h5.File('detsim.h5')

We see new data for the different nodes we've added:

In [ ]:
f['event/1/CURRENT'].keys()

Let's check out the output from drifting

In [ ]:
drifted = f['event/1/CURRENT/a_drifted/wc_deposet_to_arrow/deposet/deposet']

In [ ]:
for k in drifted.keys(): print(k)

Before we look more at the drifted info: let's look at a top-down sketch of PDHD

<img src="img/pdhd_sketch.png" style="background-color: white; padding: 10px;" />

I'd been generating the example events in the bottom left "drift volume" because, currently, the WireCell side of this tutorial has been configured to only a) drift energy depositions in the bottom 2 drift volumes and b) simulate the response on the bottom left anode. So everything we look at will be an image on the bottom left anode.


In [ ]:
#Get plotting working, and import numpy
%config InlineBackend.figure_formats = ['svg']
import matplotlib.pyplot as plt
plt.ioff()

import numpy as np

Let's check out where the drifted depositions land. First let's only look at the x position. As shown above, the x direction is parallel to the drift direction. If we define the cathode as x = 0, then the bottom left anode should be at some negative value. This is exactly what we see:

In [ ]:
plt.hist(drifted['wc.depo.x'][:])
plt.show()

We see all of the energy depositions projected onto the Y-Z plane

In [ ]:
plt.scatter(drifted['wc.depo.z'], drifted['wc.depo.y'][:])
plt.ylabel('Y (mm)')
plt.xlabel('Z (mm)')
plt.show()

We can see now that the charge depositions are now represented as arriving at some point in Y-Z at some point in time

In [ ]:
plt.hist(drifted['wc.depo.time'])
plt.show()

In [ ]:
plt.scatter(drifted['wc.depo.z'], drifted['wc.depo.time'])
plt.xlabel('Z (mm)')
plt.ylabel('time (arb units)')
plt.show()

Let's now turn to the simulated readout.

In [ ]:
sim = f['event/1/CURRENT/a_sim/wc_frame_to_arrow/frame/traces']
sim['wc.trace.charge']

The structure of this happens to be in a slightly-unfriendly format, so there's a helper script to put this into an easier format.

You first need to install a usable h5py, so in order to make a new virtual environment with h5py in it: `./make_venv.sh`

then you can run: `h5py_venv/bin/python dump_traces.py detsim.h5 -m a_sim`

Now we can look at the readout. The important part is the charge waveforms:

In [ ]:
t = np.load('detsim_a_sim_untagged.npz')
t.keys()

In [ ]:

t['event2_charge']

The rows (index 0) and columns (index 1) represent, respectively, the different electronics channels in this readout plane and clock tick according to that readout sample. 

In [ ]:
plt.imshow(t['event1_charge'], aspect='auto'); plt.colorbar()
plt.show()

We see they're at 2 different baseliens: this is becaause he indcution (U nd V) planes are a a highger baseline to account for their biolar response shape.

I'll subtract the baseline for each channel, which I take to be the median of the waveform of a channel along the full readout window.

In [ ]:
baselined = t['event2_charge'].T - np.median(t['event2_charge'], axis=1)

In [ ]:
img = plt.imshow(baselined[:, :800], aspect='auto', vmin=-100, vmax=100, origin='lower');  plt.colorbar(img, label='ADC Value')
plt.xlabel('U Plane Wire')
plt.ylabel('Tick [0.5 us]')
plt.show()

In [ ]:
img = plt.imshow(baselined[:, 800:1600], aspect='auto', vmin=-100, vmax=100, origin='lower'); plt.colorbar(img, label='ADC Value')
plt.xlabel('V Plane Wire')
plt.ylabel('Tick [0.5 us]')
plt.show()

In [ ]:
img = plt.imshow(baselined[:, -480:], aspect='auto', origin='lower', vmax=1250); plt.colorbar(img, label='ADC Value')
plt.xlabel('W Plane Wire')
plt.ylabel('Tick [0.5 us]')
plt.show()

This gives us 3 separate simulated readouts: one for each of our wire plane views (which we call U, V, and W). Note that the array slicing I did is not quite intuitive: so don't be worried if you're unsure of why I did that. I just happen to be familiar with this format

### Exercise 3: Let's discuss some qualitative aspects of this. Then: feel free to run your own simulation through detsim